<a href="https://colab.research.google.com/github/JoieLiu/PL-repo/blob/main/%E3%80%8CHW1_part2_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/11JLxQpAQmhNAs0RIjeVohcg5Ji89T9dD8gQeqDF_090/edit?usp=sharing

In [7]:
import pandas as pd
import datetime
import gradio as gr
import gspread
from google.colab import auth
from google.auth import default

In [8]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/11JLxQpAQmhNAs0RIjeVohcg5Ji89T9dD8gQeqDF_090/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"

REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人"]

_auth_done = False
_gc = None
_ws = None

In [ ]:
def _ensure_auth_and_ws():
    global _auth_done, _gc, _ws
    # 不再這裡執行 auth.authenticate_user()，因為前面手動跑過了
    if _gc is None:
        from google.auth import default
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _auth_done = True

    if _ws is None:
        # 使用你設定好的 SHEET_URL 與 WORKSHEET_NAME
        gs = _gc.open_by_url(SHEET_URL)
        _ws = gs.worksheet(WORKSHEET_NAME)
        # 確保欄位完整
        _ensure_headers()
    return _ws

def _ensure_headers():
    """確保表頭包含 REQUIRED_COLUMNS；若空表或缺欄，會補齊。"""
    rows = _ws.get_all_values()
    if not rows:
        _ws.append_row(REQUIRED_COLUMNS, value_input_option="USER_ENTERED")
        return
    header = rows[0]
    if header != REQUIRED_COLUMNS:
        # 修正：直接將第一列更新為標準表頭
        _ws.update('1:1', [REQUIRED_COLUMNS])

        # 若舊資料存在，重新整理格式
        if len(rows) > 1:
            new_rows = []
            # 建立欄位名稱對應索引
            existing = {h: idx for idx, h in enumerate(header)}
            for r in rows[1:]:
                # 根據 REQUIRED_COLUMNS 重新排列資料，找不到的補空字串
                new_row = []
                for col in REQUIRED_COLUMNS:
                    if col in existing and existing[col] < len(r):
                        new_row.append(r[existing[col]])
                    else:
                        new_row.append("")
                new_rows.append(new_row)

            # 先清掉舊資料只留新表頭
            _ws.resize(rows=1)
            # 再補回整理後的資料
            if new_rows:
                _ws.append_rows(new_rows, value_input_option="USER_ENTERED")

def _read_df():
    ws = _ensure_auth_and_ws()
    values = ws.get_all_values()
    if len(values) <= 1:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)
    df = pd.DataFrame(values[1:], columns=values[0])
    # 型別整理：確保金額是數字，方便加總
    if "金額" in df.columns:
        df["金額"] = pd.to_numeric(df["金額"], errors="coerce").fillna(0.0)
    return df

def add_expense(date_str, time_str, category, item, amount, payer):
    try:
        # 基本驗證與預設值
        if not date_str:
            date_str = datetime.date.today().strftime("%Y-%m-%d")

        category = (category or "未填").strip()
        item = (item or "未填").strip()
        payer = (payer or "匿名").strip()

        # 金額防呆轉換
        try:
            amount_val = float(amount)
        except:
            return "❌ 金額格式錯誤，請輸入數字", 0, pd.DataFrame(), pd.DataFrame()

        ws = _ensure_auth_and_ws()
        # 寫入資料到 Google Sheet
        ws.append_row(
            [date_str, time_str or "", category, item, amount_val, payer],
            value_input_option="USER_ENTERED"
        )

        msg = f"✅ 已新增：{date_str}｜{category}｜{item}｜${amount_val}"

        # 重新計算並回傳更新後的介面數據
        df = _read_df()
        cat_summary, settle_summary = _make_summary_tables(df)
        total = float(df["金額"].sum()) if not df.empty else 0.0

        return msg, total, cat_summary, settle_summary
    except Exception as e:
        return f"❌ 新增失敗：{str(e)}", 0, pd.DataFrame(), pd.DataFrame()

def refresh_summary():
    try:
        df = _read_df()
        if df.empty:
            return "目前沒有資料", 0.0, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
        total = float(df["金額"].sum())
        by_cat, settle = _make_summary_tables(df)
        return "✅ 已更新彙總", total, by_cat, settle, df
    except Exception as e:
        return f"❌ 讀取失敗：{str(e)}", 0.0, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

def _make_summary_tables(df: pd.DataFrame):
    # 分類小計邏輯
    if not df.empty and "分類" in df.columns:
        by_cat = df.groupby("分類", as_index=False)["金額"].sum().sort_values("金額", ascending=False)
    else:
        by_cat = pd.DataFrame(columns=["分類", "金額"])

    # AA 分攤邏輯
    if not df.empty and "付款人" in df.columns:
        # 處理空值並取得所有付款人
        df["付款人"] = df["付款人"].replace("", "匿名").fillna("匿名")
        unique_payers = sorted(df["付款人"].unique())
        total = df["金額"].sum()
        n = max(len(unique_payers), 1)
        aa_share = total / n

        paid_by = df.groupby("付款人", as_index=False)["金額"].sum().rename(columns={"金額": "實付"})

        # 建立完整的結算表
        settle = pd.DataFrame({"付款人": unique_payers})
        settle = settle.merge(paid_by, on="付款人", how="left").fillna(0)
        settle["應付(AA)"] = aa_share
        settle["差額(實付-應付)"] = settle["實付"] - settle["應付(AA)"]
        settle = settle.sort_values("差額(實付-應付)", ascending=False)
    else:
        settle = pd.DataFrame(columns=["付款人", "實付", "應付(AA)", "差額(實付-應付)"])

    return by_cat, settle

# 後面的 gr.Blocks 保持不變，直接啟動即可
demo.launch(share=True, debug=True, inline=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7091cfdd3dbd9aef44.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_422/392905186.py:27: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  _ws.update('1:1', [REQUIRED_COLUMNS])
